In [2]:
%pip install beautifulsoup4 lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 25.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [beautifulsoup4]

[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7"
}
BASE_URL = "https://all.rugby"

# 1. Scraping de l'effectif du Stade Toulousain pour choper les VRAIS liens
squad_url = f"{BASE_URL}/club/stade-toulousain/squad"
res = requests.get(squad_url, headers=HEADERS)

print(f"Statut de la requête effectif : {res.status_code}")

if res.status_code == 200:
    soup = BeautifulSoup(res.text, "html.parser")
    
    # Recherche de tous les liens pointant vers des fiches joueurs
    player_links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/player/" in href or "/joueur/" in href:
            if href not in player_links:
                player_links.append(href)
                
    print(f"Joueurs trouvés sur la page de l'effectif : {len(player_links)}")
    if player_links:
        print("Exemples de liens trouvés :", player_links[:3])
else:
    print("Accès bloqué ou page introuvable. Code HTTP :", res.status_code)

Statut de la requête effectif : 200
Joueurs trouvés sur la page de l'effectif : 158
Exemples de liens trouvés : ['/player/benjamin-bertrand', '/player/cyril-baille', '/player/david-ainu-u']


In [6]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7"
}
BASE_URL = "https://all.rugby"

# Liste des 14 clubs du Top 14
TOP14_CLUBS = [
    "/club/stade-toulousain",
    "/club/stade-rochelais",
    "/club/union-bordeaux-begles",
    "/club/racing-92",
    "/club/rc-toulon",
    "/club/stade-francais",
    "/club/clermont",
    "/club/castres-olympique",
    "/club/lou-rugby",
    "/club/section-paloise",
    "/club/aviron-bayonnais",
    "/club/usap",
    "/club/montpellier-herault-rugby",
    "/club/rc-vannes"
]

def collect_all_player_urls(clubs):
    player_urls = set()
    print("Collecte des joueurs sur les 14 effectifs du Top 14...")
    for club in tqdm(clubs):
        squad_url = f"{BASE_URL}{club}/squad"
        res = requests.get(squad_url, headers=HEADERS)
        if res.status_code == 200:
            soup = BeautifulSoup(res.text, "html.parser")
            for a in soup.find_all("a", href=True):
                href = a["href"]
                if "/player/" in href or "/joueur/" in href:
                    clean_href = href.replace(BASE_URL, "")
                    player_urls.add(clean_href)
        time.sleep(0.5)
    return list(player_urls)

# Exécution de la collecte des URLs
all_players_urls = collect_all_player_urls(TOP14_CLUBS)
print(f"\nTotal de joueurs uniques trouvés : {len(all_players_urls)}")
print("Exemples d'URLs :", all_players_urls[:5])

Collecte des joueurs sur les 14 effectifs du Top 14...


100%|██████████| 14/14 [00:08<00:00,  1.63it/s]


Total de joueurs uniques trouvés : 696
Exemples d'URLs : ['/player/mathis-thomas', '/player/joseph-laharrague', '/player/mateo-alquier', '/player/yanis-nama', '/player/bastien-rasal']


In [8]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7"
}
BASE_URL = "https://all.rugby"

def parse_full_player_profile(player_rel_url):
    full_url = BASE_URL + player_rel_url if player_rel_url.startswith("/") else player_rel_url
    
    try:
        res = requests.get(full_url, headers=HEADERS, timeout=10)
        if res.status_code != 200:
            return None, []
    except Exception:
        return None, []

    soup = BeautifulSoup(res.text, "html.parser")
    player_id = player_rel_url.strip("/").split("/")[-1]
    
    profile = {
        "player_id": player_id,
        "player_url": full_url,
        "name": None,
        "position": None,
        "height": None,
        "weight": None,
        "birthdate": None,
        "photo_url": None
    }

    # Nom
    h1 = soup.find("h1")
    if h1:
        profile["name"] = h1.text.strip()

    # Photo du joueur
    imgs = soup.find_all("img")
    for img in imgs:
        src = img.get("src", "")
        if "player" in src or "joueur" in src or "media" in src:
            profile["photo_url"] = src if src.startswith("http") else BASE_URL + src
            break

    # Données biométriques & Profil
    # On cherche les blocs de texte ou listes contenant taille/poids/poste
    for elem in soup.find_all(["div", "li", "p", "span"]):
        text = elem.text.strip()
        t_lower = text.lower()
        if "cm" in t_lower and len(text) < 20 and not profile["height"]:
            profile["height"] = text
        elif "kg" in t_lower and len(text) < 20 and not profile["weight"]:
            profile["weight"] = text

    # Extraction structurée des tableaux (Matchs & Infirmerie)
    logs = []
    tables = soup.find_all("table")
    for table_idx, table in enumerate(tables):
        rows = table.find_all("tr")
        for row in rows:
            cols = [td.text.strip() for td in row.find_all(["td", "th"])]
            if cols and len(cols) > 1:
                logs.append({
                    "player_id": player_id,
                    "player_name": profile["name"],
                    "table_type": "matches" if table_idx == 0 else "infirmary_or_stats",
                    "col_count": len(cols),
                    "raw_row": " | ".join(cols)
                })

    return profile, logs

# Lancement sur les 696 joueurs
profiles_list = []
logs_list = []

print(f"Lancement de l'aspiration des {len(all_players_urls)} fiches joueurs...")

for idx, rel_url in enumerate(tqdm(all_players_urls)):
    prof, logs = parse_full_player_profile(rel_url)
    if prof:
        profiles_list.append(prof)
        logs_list.extend(logs)
    
    # Pause anti-blocage IP
    time.sleep(0.4)
    
    # Sauvegarde intermédiaire tous les 50 joueurs
    if (idx + 1) % 50 == 0:
        pd.DataFrame(profiles_list).to_csv("top14_players_biometrics_checkpoint.csv", index=False)
        pd.DataFrame(logs_list).to_csv("top14_players_logs_checkpoint.csv", index=False)

# Export final complet
df_final_profiles = pd.DataFrame(profiles_list)
df_final_logs = pd.DataFrame(logs_list)

df_final_profiles.to_csv("top14_players_biometrics.csv", index=False)
df_final_logs.to_csv("top14_players_logs.csv", index=False)

print("\n Saisie globale terminée !")
print(f" Effectif total enregistré : {len(df_final_profiles)} joueurs")
print(f" Lignes de matchs / indisponibilités : {len(df_final_logs)} entrées")

Lancement de l'aspiration des 696 fiches joueurs...


100%|██████████| 696/696 [06:30<00:00,  1.78it/s]


 Saisie globale terminée !
 Effectif total enregistré : 696 joueurs
 Lignes de matchs / indisponibilités : 73725 entrées


In [12]:
import pandas as pd

df_bio = pd.read_csv("CSV/top14_players_biometrics.csv")
df_logs = pd.read_csv("CSV/top14_players_logs.csv")

print(f"Joueurs en base : {len(df_bio)}")
print(f"Lignes de logs : {len(df_logs)}\n")

print("--- APERÇU BIOMÉTRIE & PHOTOS ---")
display(df_bio.head(3))

print("\n--- APERÇU DES LOGS BRUTS (raw_row) ---")
display(df_logs[['player_name', 'table_type', 'raw_row']].head(10))

Joueurs en base : 696
Lignes de logs : 73725

--- APERÇU BIOMÉTRIE & PHOTOS ---


,player_id,player_url,name,position,height,weight,birthdate,photo_url
0,mathis-thomas,https://all.rugby/player/mathis-thomas,"Mathis THOMAS 4 years, Lock",NaN,NaN,NaN,NaN,https://all.rugby/img/joueurs/default.png
1,joseph-laharrague,https://all.rugby/player/joseph-laharrague,"Joseph LAHARRAGUE 2 years, Fly-half",NaN,NaN,NaN,NaN,https://all.rugby/img/joueurs/default.png
2,mateo-alquier,https://all.rugby/player/mateo-alquier,"Mateo ALQUIER 4 years, Hooker",NaN,NaN,NaN,NaN,https://all.rugby/img/joueurs/default.png



--- APERÇU DES LOGS BRUTS (raw_row) ---


,player_name,table_type,raw_row
0,"Mathis THOMAS 4 years, Lock",matches,Season | | Team | Tournament | Matches | W/D/...
1,"Mathis THOMAS 4 years, Lock",matches,| TOURNAMENT | Matches | W/D/L | Starter | T ...
2,"Mathis THOMAS 4 years, Lock",matches,| TEAM | Matches | W/D/L | Starter | T | D | ...
3,"Joseph LAHARRAGUE 2 years, Fly-half",matches,Season | | Team | Tournament | Matches | W/D/...
4,"Joseph LAHARRAGUE 2 years, Fly-half",matches,26/27 | | Béziers | Pro D2 | 4 | 1 0 3 | 4 | ...
5,"Joseph LAHARRAGUE 2 years, Fly-half",matches,25/26 | | Bordeaux | Top 14 | 8 | 3 0 5 | 3 |...
6,"Joseph LAHARRAGUE 2 years, Fly-half",matches,| TOURNAMENT | Matches | W/D/L | Starter | T ...
7,"Joseph LAHARRAGUE 2 years, Fly-half",matches,| Pro D2 | 4 | 1 0 3 | 4(100%) | | | 3(Avg ...
8,"Joseph LAHARRAGUE 2 years, Fly-half",matches,| Top 14 | 8 | 3 0 5 | 3(38%) | | | | | ...
9,"Joseph LAHARRAGUE 2 years, Fly-half",matches,| TEAM | Matches | W/D/L | Starter | T | D | ...


In [14]:
import requests
from bs4 import BeautifulSoup

url = "https://all.rugby/player/antoine-dupont"
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7"
}

res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

print("=== 1. TITRE ET ENTÊTE ===")
print("H1 :", soup.find("h1").text.strip() if soup.find("h1") else "Non trouvé")

print("\n=== 2. ÉLÉMENTS DE BIOMÉTRIE (Taille / Poids / Poste) ===")
# Inspection des blocs métadonnées
for div in soup.find_all(["div", "ul", "p"], class_=True):
    classes = " ".join(div.get("class", []))
    if any(k in classes for k in ["info", "player", "meta", "bio", "stat"]):
        text = div.text.strip().replace("\n", " ")
        if len(text) < 150:
            print(f"[{classes}] -> {text}")

print("\n=== 3. STRUCTURE DES TABLEAUX (Matchs / Absences) ===")
tables = soup.find_all("table")
print(f"Nombre de tableaux trouvés : {len(tables)}")

for idx, tbl in enumerate(tables):
    headers = [th.text.strip() for th in tbl.find_all("th")]
    first_row = []
    rows = tbl.find_all("tr")
    if len(rows) > 1:
        first_row = [td.text.strip() for td in rows[1].find_all(["td", "th"])]
    
    print(f"\nTableau n°{idx} (Classe: {tbl.get('class')}) :")
    print("  En-têtes :", headers)
    print("  Exemple ligne 1 :", first_row[:6])

=== 1. TITRE ET ENTÊTE ===
H1 : Antoine DUPONT  4 years, Scrum-half

=== 2. ÉLÉMENTS DE BIOMÉTRIE (Taille / Poids / Poste) ===

=== 3. STRUCTURE DES TABLEAUX (Matchs / Absences) ===
Nombre de tableaux trouvés : 27

Tableau n°0 (Classe: ['rtable', 'JOverall']) :
  En-têtes : ['Season', '', 'Team', 'Tournament', 'Matches', 'W/D/L', 'Starter', 'T', 'D', 'P', 'C', 'Points', 'Pen. Cards', 'Min', '', 'TOURNAMENT', 'Matches', 'W/D/L', 'Starter', 'T', 'D', 'P', 'C', 'Points', 'Pen. Cards', 'Min', '', 'TEAM', 'Matches', 'W/D/L', 'Starter', 'T', 'D', 'P', 'C', 'Points', 'Pen. Cards', 'Min']
  Exemple ligne 1 : ['26/27', '', 'Toulouse', 'Top 14', '4', '3 0 1']

Tableau n°1 (Classe: ['rtable', 'JSaison']) :
  En-têtes : ['', 'Tournament', 'Team', 'Opponent', 'Place', 'Res', 'Date', 'N°', 'T', 'D', 'P', 'C', 'Points', 'Pen. Cards', 'Min', '', 'Total : 5 matches', '', '', '', '', 'Starter : 80%4 / 5', '2(Avg : 0.4)', '', '', '', '10(Avg : 2.0)', '', "273'(Avg : 55')"]
  Exemple ligne 1 : ['', 'Top 1

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

url = "https://all.rugby/player/antoine-dupont"
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7"
}

res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "html.parser")

# 1. NOM & POSTE
h1_text = soup.find("h1").text.strip() if soup.find("h1") else ""
# Exemple: "Antoine DUPONT  4 years, Scrum-half"
clean_name = h1_text.split("  ")[0].strip() if "  " in h1_text else h1_text

# 2. VRAIE PHOTO OFFICIELLE (via OpenGraph meta tag)
og_image = soup.find("meta", property="og:image")
photo_url = og_image["content"] if og_image else None

# 3. TAILLE & POIDS (Recherche ciblée dans les métadonnées)
height, weight = None, None
page_text = soup.get_text()
height_match = re.search(r'(\d{1,3}\s*cm|\d,\d{2}\s*m)', page_text)
weight_match = re.search(r'(\d{2,3}\s*kg)', page_text)

if height_match: height = height_match.group(1)
if weight_match: weight = weight_match.group(1)

# 4. EXTRACTION PROPRE DES MATCHS INDIVIDUELS
clean_matches = []
tables = soup.find_all("table", class_="JSaison")

for tbl in tables:
    for row in tbl.find_all("tr"):
        cols = [td.text.strip().replace("\n", "").replace("\t", "") for td in row.find_all(["td", "th"])]
        
        # Filtre strict : un match individuel valide possède une date (ex: 06/09) en colonne 6 et des minutes en fin de ligne
        if len(cols) >= 10 and re.search(r'\d{2}/\d{2}', cols[6]):
            competition = re.sub(r'\s+', ' ', cols[1])
            club = cols[2]
            opponent = cols[3]
            venue = cols[4] # HOME / AWAY
            result = cols[5] # W / L / D
            match_date = cols[6]
            minutes_str = cols[-1] # ex: "59'"
            
            # Extraction du nombre entier de minutes
            min_match = re.search(r'(\d+)', minutes_str)
            minutes_played = int(min_match.group(1)) if min_match else 0

            clean_matches.append({
                "player_id": "antoine-dupont",
                "player_name": clean_name,
                "competition": competition,
                "club": club,
                "opponent": opponent,
                "venue": venue,
                "result": result,
                "match_date": match_date,
                "minutes_played": minutes_played
            })

# Validation des résultats
df_clean = pd.DataFrame(clean_matches)

print(f"Nom : {clean_name}")
print(f"Photo : {photo_url}")
print(f"Taille / Poids : {height} / {weight}")
print(f"\nMatchs valides extraits (sans sous-totaux) : {len(df_clean)}")
display(df_clean.head(10))

Nom : Antoine DUPONT
Photo : https://all.rugby/img/design/allrugby-square.png
Taille / Poids : None / 71 kg

Matchs valides extraits (sans sous-totaux) : 225


,player_id,player_name,competition,club,opponent,venue,result,match_date,minutes_played
0,antoine-dupont,Antoine DUPONT,Top 14 - R1,Toulouse,La Rochelle,AWAY,L,06/09,59
1,antoine-dupont,Antoine DUPONT,Top 14 - R3,Toulouse,Vannes,AWAY,W,19/09,11
2,antoine-dupont,Antoine DUPONT,Top 14 - R4,Toulouse,Montpellier,HOME,W,27/09,64
3,antoine-dupont,Antoine DUPONT,Top 14 - R6,Toulouse,Bayonne,HOME,W,11/10,80
4,antoine-dupont,Antoine DUPONT,Champions Cup - R1,Toulouse,Exeter,HOME,W,18/10,59
5,antoine-dupont,Antoine DUPONT,Top 14 - R11,Toulouse,Racing 92,HOME,L,29/11,13
6,antoine-dupont,Antoine DUPONT,Top 14 - R12,Toulouse,Lyon,AWAY,W,20/12,5
7,antoine-dupont,Antoine DUPONT,Top 14 - R13,Toulouse,La Rochelle,HOME,W,28/12,80
8,antoine-dupont,Antoine DUPONT,Champions Cup - R3,Toulouse,Saracens,AWAY,L,11/01,80
9,antoine-dupont,Antoine DUPONT,Champions Cup - R4,Toulouse,Sale,HOME,W,17/01,80


In [17]:
import time
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7"
}
BASE_URL = "https://all.rugby"

def parse_clean_player_page(player_rel_url):
    full_url = BASE_URL + player_rel_url if player_rel_url.startswith("/") else player_rel_url
    player_id = player_rel_url.strip("/").split("/")[-1]
    
    try:
        res = requests.get(full_url, headers=HEADERS, timeout=10)
        if res.status_code != 200:
            return None, []
    except Exception:
        return None, []

    soup = BeautifulSoup(res.text, "html.parser")

    # 1. NETTOYAGE NOM & POSTE
    h1_text = soup.find("h1").text.strip() if soup.find("h1") else ""
    clean_name = h1_text.split("  ")[0].strip() if "  " in h1_text else h1_text
    
    # Extraction poste
    position = None
    pos_match = re.search(r'\d+\s+years?,\s*([A-Za-z\s\-]+)', h1_text)
    if pos_match:
        position = pos_match.group(1).strip()

    # 2. VRAIE PHOTO JOUEUR
    photo_url = None
    img_tag = soup.find("img", src=re.compile(r'/joueurs/|/players/|/media/'))
    if img_tag and "src" in img_tag.attrs:
        src = img_tag["src"]
        photo_url = src if src.startswith("http") else BASE_URL + src

    # 3. TAILLE & POIDS
    page_text = soup.get_text()
    height_match = re.search(r'(\d{1,3}\s*cm|\d,\d{2}\s*m)', page_text)
    weight_match = re.search(r'(\d{2,3}\s*kg)', page_text)
    
    height = height_match.group(1) if height_match else None
    weight = weight_match.group(1) if weight_match else None

    profile_data = {
        "player_id": player_id,
        "clean_name": clean_name,
        "position": position,
        "height": height,
        "weight": weight,
        "photo_url": photo_url,
        "player_url": full_url
    }

    # 4. PARSING DES MATCHS INDIVIDUELS (AVEC ANNEE RECONSTITUÉE)
    matches_data = []
    tables = soup.find_all("table", class_="JSaison")

    for tbl in tables:
        # Recherche de la saison (ex: 2023-2024 ou 23/24) au-dessus du tableau
        season_str = "2024"
        prev_elem = tbl.find_previous(["h2", "h3", "div", "span"])
        if prev_elem:
            s_match = re.search(r'(\d{2,4})[/|-](\d{2,4})', prev_elem.text)
            if s_match:
                season_str = s_match.group(1)
                if len(season_str) == 2:
                    season_str = "20" + season_str

        for row in tbl.find_all("tr"):
            cols = [td.text.strip().replace("\n", "").replace("\t", "") for td in row.find_all(["td", "th"])]
            
            # Filtre strict : présence d'une date (DD/MM) en colonne 6 et minutes en fin de ligne
            if len(cols) >= 10 and re.search(r'\d{2}/\d{2}', cols[6]):
                competition = re.sub(r'\s+', ' ', cols[1])
                club = cols[2]
                opponent = cols[3]
                venue = cols[4]
                result = cols[5]
                raw_date = cols[6] # ex: 06/09
                
                # Formatage de la date en AAAA-MM-JJ
                day, month = raw_date.split("/")
                year = season_str
                # Ajustement d'année pour la 2ème moitié de saison (Janvier à Juin)
                if int(month) < 7:
                    year = str(int(season_str) + 1)
                full_date = f"{year}-{month.zfill(2)}-{day.zfill(2)}"

                # Core minutes played
                min_match = re.search(r'(\d+)', cols[-1])
                minutes_played = int(min_match.group(1)) if min_match else 0

                matches_data.append({
                    "player_id": player_id,
                    "player_name": clean_name,
                    "competition": competition,
                    "club": club,
                    "opponent": opponent,
                    "venue": venue,
                    "result": result,
                    "match_date": full_date,
                    "minutes_played": minutes_played
                })

    return profile_data, matches_data

# Lancement de la génération du dataset de production
all_profiles = []
all_matches = []

print(f"Traitements de la base complète ({len(all_players_urls)} joueurs)...")
for idx, rel_url in enumerate(tqdm(all_players_urls)):
    prof, match_list = parse_clean_player_page(rel_url)
    if prof:
        all_profiles.append(prof)
        all_matches.extend(match_list)
    
    time.sleep(0.3)
    
    # Checkpoint tous les 100 joueurs
    if (idx + 1) % 100 == 0:
        pd.DataFrame(all_profiles).to_csv("CSV/top14_clean_players.csv", index=False)
        pd.DataFrame(all_matches).to_csv("CSV/top14_clean_matches.csv", index=False)

# Sauvegarde finale
df_clean_players = pd.DataFrame(all_profiles)
df_clean_matches = pd.DataFrame(all_matches)

df_clean_players.to_csv("CSV/top14_clean_players.csv", index=False)
df_clean_matches.to_csv("CSV/top14_clean_matches.csv", index=False)

print("\n Base nettoyée générée avec succès !")
print(f" Profils valides : {len(df_clean_players)}")
print(f" Matchs structurés enregistrés : {len(df_clean_matches)}")

Traitements de la base complète (696 joueurs)...


100%|██████████| 696/696 [05:23<00:00,  2.15it/s]


 Base nettoyée générée avec succès !
 Profils valides : 696
 Matchs structurés enregistrés : 38475


In [18]:
import pandas as pd
import numpy as np

df_players = pd.read_csv("CSV/top14_clean_players.csv")
df_matches = pd.read_csv("CSV/top14_clean_matches.csv")

print("==================================================")
print("       BILAN DE QUALITÉ DE LA DATASET TOP 14      ")
print("==================================================\n")

# 1. PROFONDEUR TEMPORELLE
df_matches['match_date'] = pd.to_datetime(df_matches['match_date'])
min_date = df_matches['match_date'].min().strftime('%Y-%m-%d')
max_date = df_matches['match_date'].max().strftime('%Y-%m-%d')
print(f"1. COUVERTURE TEMPORELLE : Du {min_date} au {max_date}")

# 2. VOLUMÉTRIE PAR JOUEUR
matches_per_player = df_matches.groupby('player_id').size()
print(f"2. MATCHS PAR JOUEUR :")
print(f"   - Moyenne : {matches_per_player.mean():.1f} matchs / joueur")
print(f"   - Médiane : {matches_per_player.median():.0f} matchs / joueur")
print(f"   - Joueurs avec < 5 matchs : {(matches_per_player < 5).sum()} (à surveiller/filtrer)")
print(f"   - Joueurs avec > 50 matchs : {(matches_per_player >= 50).sum()} (solide historique)")

# 3. DISTRIBUTION DU TEMPS DE JEU
print(f"\n3. RÉPARTITION DU TEMPS DE JEU (Minutes) :")
print(f"   - Matchs complets (>= 70 min) : {(df_matches['minutes_played'] >= 70).sum()}")
print(f"   - Entrées en jeu (< 30 min)   : {((df_matches['minutes_played'] > 0) & (df_matches['minutes_played'] < 30)).sum()}")
print(f"   - Matchs à 0 min enregistrés  : {(df_matches['minutes_played'] == 0).sum()}")

# 4. QUALITÉ DES PROFILS
print(f"\n4. COMPLÉTUDE DES PROFILS JOUEURS ({len(df_players)} total) :")
print(f"   - Noms valides     : {df_players['clean_name'].notna().sum()}")
print(f"   - Postes renseignés : {df_players['position'].notna().sum()}")
print(f"   - Photos récupérées : {df_players['photo_url'].notna().sum()}")

# 5. NOMBRE DE CLUBS DISTINCTS
print(f"\n5. CLUBS ENREGISTRÉS : {df_matches['club'].nunique()} clubs")
print("   Exemples :", list(df_matches['club'].unique()[:5]))

       BILAN DE QUALITÉ DE LA DATASET TOP 14      

1. COUVERTURE TEMPORELLE : Du 2013-07-04 au 2026-10-24
2. MATCHS PAR JOUEUR :
   - Moyenne : 89.9 matchs / joueur
   - Médiane : 84 matchs / joueur
   - Joueurs avec < 5 matchs : 47 (à surveiller/filtrer)
   - Joueurs avec > 50 matchs : 275 (solide historique)

3. RÉPARTITION DU TEMPS DE JEU (Minutes) :
   - Matchs complets (>= 70 min) : 10926
   - Entrées en jeu (< 30 min)   : 12917
   - Matchs à 0 min enregistrés  : 0

4. COMPLÉTUDE DES PROFILS JOUEURS (696 total) :
   - Noms valides     : 696
   - Postes renseignés : 692
   - Photos récupérées : 690

5. CLUBS ENREGISTRÉS : 151 clubs
   Exemples : ['Béziers', 'Bordeaux', 'Bayonne', 'France U20', 'Clermont']


In [20]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from tqdm import tqdm

df_players = pd.read_csv("CSV/top14_clean_players.csv")

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def extract_bio_from_text(player_url):
    try:
        res = requests.get(player_url, headers=headers, timeout=8)
        if res.status_code != 200:
            return None, None
        
        soup = BeautifulSoup(res.text, "html.parser")
        text = soup.get_text()

        # Regex ciblées sur la phrase Bio
        # "standing at 1.91 m tall" -> 191
        h_match = re.search(r'standing at (\d[.,]\d{2})\s*m', text, re.IGNORECASE)
        # "weighing in at 113 kg" -> 113
        w_match = re.search(r'weighing in at (\d{2,3})\s*kg', text, re.IGNORECASE)

        height_cm = int(float(h_match.group(1).replace(',', '.')) * 100) if h_match else None
        weight_kg = int(w_match.group(1)) if w_match else None

        return height_cm, weight_kg
    except Exception:
        return None, None

# Test rapide sur un échantillon pour valider
print("Extraction des vraies données biométriques...")
results = []
for idx, row in tqdm(df_players.iterrows(), total=len(df_players)):
    h_cm, w_kg = extract_bio_from_text(row['player_url'])
    results.append({
        'player_id': row['player_id'],
        'clean_name': row['clean_name'],
        'height_cm': h_cm,
        'weight_kg': w_kg
    })

df_bio_real = pd.DataFrame(results)

print("\n=== BILAN EXTRACTION RÉELLE ===")
print(f"Tailles réelles extraites : {df_bio_real['height_cm'].notna().sum()} / {len(df_bio_real)}")
print(f"Poids réels extraits   : {df_bio_real['weight_kg'].notna().sum()} / {len(df_bio_real)}")

display(df_bio_real.dropna(subset=['height_cm']).head(10))

Extraction des vraies données biométriques...


100%|██████████| 696/696 [01:31<00:00,  7.57it/s]


=== BILAN EXTRACTION RÉELLE ===
Tailles réelles extraites : 474 / 696
Poids réels extraits   : 525 / 696


,player_id,clean_name,height_cm,weight_kg
0,mathis-thomas,Mathis THOMAS,191.0,99.0
1,joseph-laharrague,Joseph LAHARRAGUE,177.0,69.0
4,bastien-rasal,Bastien RASAL,178.0,74.0
6,lachlan-swinton,Lachlan SWINTON,189.0,99.0
7,enzo-fluxa,Enzo FLUXA,173.0,85.0
9,jules-said,Jules SAÏD,181.0,88.0
12,timeo-frier,Timéo FRIER,182.0,83.0
13,joaquin-oviedo,Joaquín OVIEDO,181.0,103.0
14,zinedine-aouad,Zinédine AOUAD,185.0,111.0
15,christian-everitt,Christian EVERITT,184.0,99.0


In [25]:
import pandas as pd

# 1. Chargement de la base d'origine des joueurs
df_players_clean = pd.read_csv("CSV/top14_clean_players.csv")
df_matches = pd.read_csv("CSV/top14_clean_matches.csv")

# 2. Fusion avec les biométries réelles extraites (df_bio_real)
try:
    # Si df_bio_real est en mémoire, on met à jour les colonnes
    bio_data = df_bio_real[['player_id', 'height_cm', 'weight_kg']]
    df_players = pd.merge(
        df_players_clean.drop(columns=['height', 'weight'], errors='ignore'),
        bio_data,
        on='player_id',
        how='left'
    )
except NameError:
    # Si le notebook a redémarré, on utilise la base propre
    df_players = df_players_clean.copy()
    df_players['weight_kg'] = df_players['weight'].str.extract(r'(\d+)').astype(float)
    df_players['height_cm'] = None

# Sauvegarde de la base profils consolidée
df_players.to_csv("CSV/top14_players_real_bio.csv", index=False)

print("=== 1. AUDIT DES VALEURS MANQUANTES (PROFILS CONSOLIDES) ===")
cols_to_check = [c for c in ['position', 'height_cm', 'weight_kg'] if c in df_players.columns]
print(df_players[cols_to_check].isna().sum().to_frame(name='NaN_count'))

print("\n=== 2. AUDIT DES VALEURS MANQUANTES (MATCHS) ===")
print(df_matches[['player_id', 'match_date', 'minutes_played', 'club', 'competition']].isna().sum().to_frame(name='NaN_count'))

# 3. Répartition par volume d'historique
matches_per_player = df_matches.groupby('player_id').size()

less_than_10 = (matches_per_player < 10).sum()
ge_10 = (matches_per_player >= 10).sum()

print("\n=== 3. FILTRAGE D'HISTORIQUE ===")
print(f"Joueurs avec < 10 matchs (à exclure)  : {less_than_10}")
print(f"Joueurs avec >= 10 matchs (à conserver) : {ge_10}")

=== 1. AUDIT DES VALEURS MANQUANTES (PROFILS CONSOLIDES) ===
           NaN_count
position           4
height_cm        222
weight_kg        171

=== 2. AUDIT DES VALEURS MANQUANTES (MATCHS) ===
                NaN_count
player_id               0
match_date              0
minutes_played          0
club                    0
competition             0

=== 3. FILTRAGE D'HISTORIQUE ===
Joueurs avec < 10 matchs (à exclure)  : 74
Joueurs avec >= 10 matchs (à conserver) : 354


In [26]:
import pandas as pd

# 1. Chargement des bases
df_players = pd.read_csv("CSV/top14_players_real_bio.csv")
df_matches = pd.read_csv("CSV/top14_clean_matches.csv")

# 2. Identification des 354 joueurs avec un historique exploitable (>= 10 matchs)
matches_per_player = df_matches.groupby('player_id').size()
valid_player_ids = matches_per_player[matches_per_player >= 10].index

# 3. Filtrage strict des deux tables
df_matches_gold = df_matches[df_matches['player_id'].isin(valid_player_ids)].copy()
df_players_gold = df_players[df_players['player_id'].isin(valid_player_ids)].copy()

# 4. Traitement du poste pour les 4 NaN
df_players_gold['position'] = df_players_gold['position'].fillna('Inconnu')

# 5. Sauvegarde des fichiers Gold
df_players_gold.to_csv("CSV/top14_gold_players.csv", index=False)
df_matches_gold.to_csv("CSV/top14_gold_matches.csv", index=False)

print("=== DATASET GOLD SCELLÉ ===")
print(f"Joueurs conservés (≥ 10 matchs) : {len(df_players_gold)}")
print(f"Matchs conservés               : {len(df_matches_gold)}")
print(f"Saisons couvertes              : {df_matches_gold['match_date'].str[:4].min()} à {df_matches_gold['match_date'].str[:4].max()}")

=== DATASET GOLD SCELLÉ ===
Joueurs conservés (≥ 10 matchs) : 354
Matchs conservés               : 38184
Saisons couvertes              : 2013 à 2026


In [27]:
import pandas as pd
import numpy as np

df_matches = pd.read_csv("CSV/top14_gold_matches.csv")
df_matches['match_date'] = pd.to_datetime(df_matches['match_date'])
df_matches = df_matches.sort_values(by=['player_id', 'match_date']).reset_index(drop=True)

int_competitions = ['Six Nations', 'World Cup', 'Test Match', 'Autumn Nations', 'Challenge Cup']

df_matches['days_since_last_match'] = df_matches.groupby('player_id')['match_date'].diff().dt.days

def flag_realistic_injuries(df):
    is_injured_list = []
    
    for i in range(len(df)):
        gap = df.iloc[i]['days_since_last_match']
        
        if pd.isna(gap):
            is_injured_list.append(0)
            continue
            
        # 1. Écart compatible avec une blessure modérée/sévère (3 à 12 semaines)
        is_suspicious_gap = 21 < gap <= 90
        
        # 2. Vérification des sélections internationales
        curr_comp = str(df.iloc[i]['competition'])
        prev_comp = str(df.iloc[i-1]['competition']) if i > 0 else ""
        is_international = any(c in curr_comp or c in prev_comp for c in int_competitions)
        
        # 3. Filtre de présence préalable : le joueur a-t-il joué au moins 60 min sur ses 2 matchs précédents ?
        prior_minutes = 0
        if i >= 2:
            prior_minutes = df.iloc[i-1]['minutes_played'] + df.iloc[i-2]['minutes_played']
        elif i == 1:
            prior_minutes = df.iloc[i-1]['minutes_played']
            
        was_active_player = prior_minutes >= 60
        
        # Validation de la cible Y
        if is_suspicious_gap and not is_international and was_active_player:
            is_injured_list.append(1)
        else:
            is_injured_list.append(0)
            
    df['is_injury_return'] = is_injured_list
    return df

df_processed = df_matches.groupby('player_id', group_keys=False).apply(flag_realistic_injuries)

# Sauvegarde
df_processed.to_csv("CSV/top14_matches_processed.csv", index=False)

print("=== VARIABLE CIBLE Y ASSAINIE ===")
print(f"Total des feuilles de match : {len(df_processed)}")
print(f"Événements retenus (Y = 1)  : {df_processed['is_injury_return'].sum()}")
print(f"Taux de positifs          : {df_processed['is_injury_return'].mean():.2%}")

=== VARIABLE CIBLE Y ASSAINIE ===
Total des feuilles de match : 38184
Événements retenus (Y = 1)  : 4404
Taux de positifs          : 11.53%


/var/folders/10/drbnlq896kg8_yg39l1dgjh40000gn/T/ipykernel_54870/3597957864.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_processed = df_matches.groupby('player_id', group_keys=False).apply(flag_realistic_injuries)


In [28]:
# Vérification des anomalies structurelles
print("Doublons de dates pour un même joueur :", df_processed.duplicated(subset=['player_id', 'match_date']).sum())
print("Matchs avec minutes incohérentes    :", (df_processed['minutes_played'] < 0).sum())
print("Plage temporelle couverte             :", df_processed['match_date'].min().strftime('%Y-%m-%d'), "au", df_processed['match_date'].max().strftime('%Y-%m-%d'))


Doublons de dates pour un même joueur : 7
Matchs avec minutes incohérentes    : 0
Plage temporelle couverte             : 2013-07-04 au 2026-10-24


In [29]:
import pandas as pd

df_processed = pd.read_csv("CSV/top14_matches_processed.csv")
df_processed['match_date'] = pd.to_datetime(df_processed['match_date'])

# Tri par joueur, date et minutes dégressives
df_processed = df_processed.sort_values(
    by=['player_id', 'match_date', 'minutes_played'], 
    ascending=[True, True, False]
)

# Suppression des doublons stricts sur le couple (player_id, match_date)
df_processed = df_processed.drop_duplicates(subset=['player_id', 'match_date'], keep='first').reset_index(drop=True)

# Sauvegarde
df_processed.to_csv("CSV/top14_matches_processed.csv", index=False)

print("=== VERIFICATION DU NETTOYAGE ===")
print("Doublons restants :", df_processed.duplicated(subset=['player_id', 'match_date']).sum())
print("Total final des matchs :", len(df_processed))

=== VERIFICATION DU NETTOYAGE ===
Doublons restants : 0
Total final des matchs : 38177
